In [ ]:
import os
import sys

# Otteniamo il percorso assoluto della cartella corrente in cui Python sta eseguendo
current_dir = os.getcwd()

# Se il path finisce con 'notebooks', stiamo eseguendo in locale dentro la sottocartella.
# Dobbiamo salire di un livello per arrivare alla root del progetto (CLARITY)
if current_dir.endswith('notebooks'):
    project_root = os.path.abspath(os.path.join(current_dir, '..'))
else:
    # Se NON finisce con 'notebooks', siamo già nella root (es. script di setup su Colab)
    project_root = current_dir

# Aggiungiamo la root al sys.path solo se non c'è già
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import config
from src.dataset import ClarityEvasionDataset

import torch
import torch.nn as nn
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from sklearn.metrics import f1_score, precision_recall_fscore_support
import numpy as np
import pickle

print(f"[*] MODALITÀ DEBUG ATTIVA? : {config.DEBUG_MODE}")
print(f"[*] MODELLO SELEZIONATO    : {config.MODEL_NAME}")
print(f"[*] BATCH SIZE IMPOSTATO   : {config.BATCH_SIZE}")
print(f"[*] EPOCHE IMPOSTATE       : {config.EPOCHS}")

# Device check
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[*] DEVICE DISPONIBILE     : {device.type.upper()}")

e:\Riky\Documents\Poli\CLARITY\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[*] MODALITÀ DEBUG ATTIVA? : True
[*] MODELLO SELEZIONATO    : prajjwal1/bert-tiny
[*] BATCH SIZE IMPOSTATO   : 2
[*] EPOCHE IMPOSTATE       : 1
[*] DEVICE DISPONIBILE     : CUDA


In [2]:
train_path = os.path.join(config.DATA_DIR, "train_dataset.pkl")
val_path = os.path.join(config.DATA_DIR, "val_dataset.pkl")

print("\n[*] Caricamento dei Dataset da disco...")
with open(train_path, 'rb') as f:
    train_dataset = pickle.load(f)

with open(val_path, 'rb') as f:
    val_dataset = pickle.load(f)

print(f"Train Dataset size : {len(train_dataset)}")
print(f"Validation Dataset size: {len(val_dataset)}")


[*] Caricamento dei Dataset da disco...
Train Dataset size : 40
Validation Dataset size: 10


In [3]:
from collections import Counter

print("\n[*] Calcolo dei pesi delle classi per contrastare lo sbilanciamento...")

# Estraiamo tutte le etichette dal dataset di training
all_train_labels = [item['labels'].item() for item in train_dataset]
class_counts = Counter(all_train_labels)

num_classes = len(config.EVASION_LABELS)
total_samples = len(all_train_labels)

# Calcoliamo i pesi (Formula bilanciata Sklearn)
# weight = n_samples / (n_classes * np.bincount(y))
class_weights = []
for i in range(num_classes):
    # Gestiamo il caso (rarissimo in debug) in cui una classe ha 0 esempi per evitare divisioni per zero
    count = class_counts.get(i, 1) 
    weight = total_samples / (num_classes * count)
    class_weights.append(weight)

# Trasformiamo in un tensore PyTorch e spostiamolo sulla GPU (o CPU)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

print("\nPesi calcolati per classe (Maggiore peso = Classe più rara = Più penalità se sbagliata):")
for i, w in enumerate(class_weights):
    print(f" - {config.ID2EVASION[i]:<20}: {w:.4f}")


[*] Calcolo dei pesi delle classi per contrastare lo sbilanciamento...

Pesi calcolati per classe (Maggiore peso = Classe più rara = Più penalità se sbagliata):
 - Explicit            : 0.3175
 - Implicit            : 1.1111
 - Dodging             : 0.6349
 - General             : 1.4815
 - Deflection          : 0.8889
 - Partial/half-answer : 4.4444
 - Declining to answer : 0.8889
 - Claims ignorance    : 4.4444
 - Clarification       : 2.2222


In [4]:
class WeightedTrainer(Trainer):
    """
    Sottoclasse del Trainer HuggingFace per supportare la CrossEntropyLoss con i pesi di classe.
    Questo impedisce al modello di "pigrire" sulle classi maggioritarie (es. Explicit).
    """
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.pop("labels")
        # Forward pass
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        # Istanziamo la Loss passando i pesi calcolati prima
        # Il tensore 'class_weights_tensor' è globale nella cella precedente.
        loss_fct = nn.CrossEntropyLoss(weight=class_weights_tensor)
        
        # Calcoliamo la loss
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    """
    Calcola le metriche alla fine di ogni epoca di validazione.
    Ritorna la Macro F1 (metrica ufficiale della gara SemEval) e l'Accuracy.
    """
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    # average='macro' assicura che le classi piccole valgano quanto le grandi
    macro_f1 = f1_score(labels, predictions, average='macro', zero_division=0)
    
    # Per completezza, calcoliamo anche la precision/recall/f1 pesate sulla distribuzione
    precision, recall, f1_weighted, _ = precision_recall_fscore_support(
        labels, predictions, average='weighted', zero_division=0
    )
    
    return {
        "macro_f1": macro_f1,
        "weighted_f1": f1_weighted, # Simile all'accuracy, utile per capire se overfittiamo le classi grandi
    }

In [5]:
print(f"\n[*] Istanziazione del Modello: {config.MODEL_NAME}")

# Carichiamo il modello con 9 classi di output
model = AutoModelForSequenceClassification.from_pretrained(
    config.MODEL_NAME,
    num_labels=len(config.EVASION_LABELS),
    id2label=config.ID2EVASION,
    label2id=config.EVASION2ID
)

# Definiamo i parametri di addestramento
# Qui simuliamo un batch più grande (se siamo in Colab) usando gradient accumulation.
grad_accum_steps = 1 if config.DEBUG_MODE else 4

training_args = TrainingArguments(
    output_dir=config.MODELS_DIR,         # Dove salvare i checkpoints
    num_train_epochs=config.EPOCHS,
    per_device_train_batch_size=config.BATCH_SIZE,
    per_device_eval_batch_size=config.BATCH_SIZE,
    gradient_accumulation_steps=grad_accum_steps,
    learning_rate=config.LEARNING_RATE,
    weight_decay=0.01,                    # Aiuta a prevenire l'overfitting
    evaluation_strategy="epoch",          # Valutiamo alla fine di ogni epoca
    save_strategy="epoch",                # Salviamo il modello alla fine di ogni epoca
    logging_dir=os.path.join(config.ROOT_DIR, "logs"),
    logging_steps=10,
    load_best_model_at_end=True,          # Importante: a fine training ricarica il modello con la metrica migliore
    metric_for_best_model="macro_f1",     # Ottimizziamo la Macro F1 (NON la loss!)
    greater_is_better=True,               # F1 più alta è meglio
    report_to="none"                      # Evita di stampare log verso wandb/tensorboard (per ora)
)

# Istanziamo il nostro Custom Trainer
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)] # Si ferma se per 2 epoche la F1 non sale
)

print("\n[*] Inizio del Training...")
trainer.train()

print("\n[✓] Training completato. Il modello migliore è stato caricato in RAM.")


[*] Istanziazione del Modello: prajjwal1/bert-tiny


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at prajjwal1/bert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
e:\Riky\Documents\Poli\CLARITY\venv\Lib\site-packages\transformers\training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(



[*] Inizio del Training...


 90%|█████████ | 18/20 [00:00<00:00, 33.30it/s]

{'loss': 2.1416, 'grad_norm': 10.689062118530273, 'learning_rate': 1e-05, 'epoch': 0.5}


100%|██████████| 20/20 [00:00<00:00, 33.30it/s]

{'loss': 2.1609, 'grad_norm': 7.597785949707031, 'learning_rate': 0.0, 'epoch': 1.0}


                                               
100%|██████████| 20/20 [00:00<00:00, 20.57it/s]

{'eval_loss': 2.2243759632110596, 'eval_macro_f1': 0.0, 'eval_weighted_f1': 0.0, 'eval_runtime': 0.0617, 'eval_samples_per_second': 162.069, 'eval_steps_per_second': 81.035, 'epoch': 1.0}
{'train_runtime': 0.9747, 'train_samples_per_second': 41.037, 'train_steps_per_second': 20.519, 'train_loss': 2.1512255668640137, 'epoch': 1.0}

[✓] Training completato. Il modello migliore è stato caricato in RAM.


In [6]:
final_model_path = os.path.join(config.MODELS_DIR, "best_evasion_encoder")

print(f"\n[*] Salvataggio del modello migliore e del tokenizer in: {final_model_path}")

trainer.save_model(final_model_path)
tokenizer = train_dataset.tokenizer
tokenizer.save_pretrained(final_model_path)

print("\n[✓] Salvataggio Completato! Sei pronto per il Notebook 03_Evaluation_and_Analysis.ipynb.")


[*] Salvataggio del modello migliore e del tokenizer in: e:\Riky\Documents\Poli\CLARITY\notebooks\saved_models\best_evasion_encoder

[✓] Salvataggio Completato! Sei pronto per il Notebook 03_Evaluation_and_Analysis.ipynb.
